In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings("ignore")

In [2]:
movies = pd.read_csv("/content/movies.csv")
ratings = pd.read_csv("/content/ratings.csv")

In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
ratings.head()

In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [6]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [7]:
ratings.isnull().sum()


,0
userId,0
movieId,0
rating,0
timestamp,0


In [9]:
user_movie_matrix = ratings.pivot_table(
        index='userId',
        columns='movieId',
        values='rating'
)

In [11]:
user_movie_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,2.5,NaN,NaN,NaN,NaN,NaN,2.5,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,2.5,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
user_movie_matrix = user_movie_matrix.fillna(0)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

In [17]:
similar_users = user_similarity_df[1].sort_values(ascending=False)

print(similar_users.head(10))

userId
1      1.000000
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
469    0.330664
39     0.329782
288    0.329700
452    0.328048
Name: 1, dtype: float64


In [18]:
similar_users = similar_users.iloc[1:6]

print(similar_users)

userId
266    0.357408
313    0.351562
368    0.345127
57     0.345034
91     0.334727
Name: 1, dtype: float64


In [19]:
watched_movies = user_movie_matrix.loc[1]

watched_movies = watched_movies[watched_movies > 0]

print(watched_movies.head())

movieId
1     4.0
3     4.0
6     4.0
47    5.0
50    5.0
Name: 1, dtype: float64


In [20]:
recommendations = {}

for sim_user in similar_users.index:

    sim_user_ratings = user_movie_matrix.loc[sim_user]

    for movie, rating in sim_user_ratings.items():

        if rating > 0 and movie not in watched_movies.index:

            recommendations[movie] = recommendations.get(movie, 0) + rating

In [21]:
len(recommendations)

997

In [22]:
recommendations = sorted(
    recommendations.items(),
    key=lambda x: x[1],
    reverse=True
)

recommendations[:10]

[(1200, 24.0),
 (1610, 21.5),
 (541, 20.0),
 (589, 20.0),
 (1036, 20.0),
 (858, 20.0),
 (924, 19.5),
 (1374, 19.5),
 (2791, 17.5),
 (1221, 17.0)]

In [23]:
movie_titles = movies.set_index("movieId")["title"].to_dict()

In [24]:
print("Top 5 Recommendations:\n")

for movie_id, score in recommendations[:5]:

    print(movie_titles[movie_id], " | Score:", score)

Top 5 Recommendations:

Aliens (1986)  | Score: 24.0
Hunt for Red October, The (1990)  | Score: 21.5
Blade Runner (1982)  | Score: 20.0
Terminator 2: Judgment Day (1991)  | Score: 20.0
Die Hard (1988)  | Score: 20.0


In [25]:
def recommend_movies(user_id, top_n=5):

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    similar_users = similar_users.iloc[1:6]

    watched_movies = user_movie_matrix.loc[user_id]

    watched_movies = watched_movies[watched_movies > 0]

    recommendations = {}

    for sim_user in similar_users.index:

        sim_user_ratings = user_movie_matrix.loc[sim_user]

        for movie, rating in sim_user_ratings.items():

            if rating > 0 and movie not in watched_movies.index:

                recommendations[movie] = recommendations.get(movie, 0) + rating

    recommendations = sorted(
        recommendations.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nTop {top_n} recommendations for User {user_id}:\n")

    for movie_id, score in recommendations[:top_n]:

        print(f"{movie_titles[movie_id]} (Score: {score:.2f})")

In [26]:
movie_user_matrix = user_movie_matrix.T

movie_user_matrix.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,0.0,0.0,4.0,0.0,4.5,0.0,0.0,0.0,...,4.0,0.0,4.0,3.0,4.0,2.5,4.0,2.5,3.0,5.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,...,0.0,4.0,0.0,5.0,3.5,0.0,0.0,2.0,0.0,0.0
3,4.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(movie_user_matrix)

In [28]:
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=movie_user_matrix.index,
    columns=movie_user_matrix.index
)

item_similarity_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.410562,0.296917,0.035573,0.308762,0.376316,0.277491,0.131629,0.232586,0.395573,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.410562,1.000000,0.282438,0.106415,0.287795,0.297009,0.228576,0.172498,0.044835,0.417693,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.296917,0.282438,1.000000,0.092406,0.417802,0.284257,0.402831,0.313434,0.304840,0.242954,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.035573,0.106415,0.092406,1.000000,0.188376,0.089685,0.275035,0.158022,0.000000,0.095598,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.308762,0.287795,0.417802,0.188376,1.000000,0.298969,0.474002,0.283523,0.335058,0.218061,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
similar_movies = item_similarity_df[1].sort_values(ascending=False)

similar_movies.head(10)

,1
movieId,
1,1.000000
3114,0.572601
480,0.565637
780,0.564262
260,0.557388
356,0.547096
364,0.541145
1210,0.541089
648,0.538913


In [30]:
similar_movies = similar_movies.iloc[1:6]

print(similar_movies)

movieId
3114    0.572601
480     0.565637
780     0.564262
260     0.557388
356     0.547096
Name: 1, dtype: float64


In [31]:
movie_titles = movies.set_index("movieId")["title"].to_dict()

In [32]:
print("Movies similar to:", movie_titles[1])

for movie_id, similarity in similar_movies.items():

    print(movie_titles[movie_id], similarity)

Movies similar to: Toy Story (1995)
Toy Story 2 (1999) 0.5726012603197153
Jurassic Park (1993) 0.5656368040861564
Independence Day (a.k.a. ID4) (1996) 0.5642616935276658
Star Wars: Episode IV - A New Hope (1977) 0.5573881705799365
Forrest Gump (1994) 0.547095907940174


In [33]:
def item_based_recommendation(user_id, top_n=5):

    user_ratings = user_movie_matrix.loc[user_id]

    watched_movies = user_ratings[user_ratings > 0]

    scores = {}

    for movie_id in watched_movies.index:

        similar_movies = item_similarity_df[movie_id]

        for sim_movie, similarity in similar_movies.items():

            if sim_movie not in watched_movies.index:

                scores[sim_movie] = scores.get(sim_movie, 0) + similarity

    scores = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    print(f"\nTop {top_n} recommendations for User {user_id}:\n")

    for movie_id, score in scores[:top_n]:

        print(f"{movie_titles[movie_id]} (Similarity Score: {score:.3f})")

In [34]:
item_based_recommendation(10)


Top 5 recommendations for User 10:

Pirates of the Caribbean: The Curse of the Black Pearl (2003) (Similarity Score: 39.596)
Hangover, The (2009) (Similarity Score: 38.269)
Ocean's Eleven (2001) (Similarity Score: 38.260)
Harry Potter and the Goblet of Fire (2005) (Similarity Score: 38.092)
Wedding Crashers (2005) (Similarity Score: 37.803)


In [35]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print(train_data.shape)
print(test_data.shape)

(80668, 4)
(20168, 4)


In [36]:
train_matrix = train_data.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

train_similarity = cosine_similarity(train_matrix)

train_similarity_df = pd.DataFrame(
    train_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

In [38]:
import numpy as np

def predict_rating(user_id, movie_id):

    # User or movie missing in training data
    if user_id not in train_matrix.index:
        return 0

    if movie_id not in train_matrix.columns:
        return 0

    # Similarity scores
    similarities = train_similarity_df[user_id]

    # Ratings for the movie
    movie_ratings = train_matrix[movie_id]

    # Only users who rated the movie
    mask = movie_ratings > 0

    similarities = similarities[mask]
    movie_ratings = movie_ratings[mask]

    if len(movie_ratings) == 0:
        return 0

    predicted = np.dot(similarities, movie_ratings) / similarities.sum()

    return predicted

In [39]:
actual = []
predicted = []

sample_test = test_data.sample(1000, random_state=42)

for _, row in sample_test.iterrows():

    pred = predict_rating(row['userId'], row['movieId'])

    if pred > 0:

        predicted.append(pred)
        actual.append(row['rating'])

In [40]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(actual, predicted))

print("RMSE:", rmse)

RMSE: 0.9672935717512379


In [41]:
score = similarity * rating

In [42]:
import pickle

model_data = {
    "user_movie_matrix": user_movie_matrix,
    "user_similarity_df": user_similarity_df,
    "item_similarity_df": item_similarity_df,
    "movie_titles": movie_titles
}

with open("movie_recommender.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("Model saved successfully!")

Model saved successfully!
